# Finance Data Loading

This notebook loads all finance-related CSV files from three product line directories into Delta tables.

## Data Sources:
- **Camping**: Files/data/camping/finance/
- **Kitchen**: Files/data/kitchen/finance/
- **Ski**: Files/data/ski/finance/

## Tables to Load:
1. **Invoice** - Customer invoices from all three product lines
2. **Account** - Financial accounts from all three product lines  
3. **Payment** - Customer payments from all three product lines

In [ ]:
# Setup and Configuration
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Schema Configuration
SCHEMA_NAME = "finance"
BASE_PATH = "Files/data"

# Product line paths
PRODUCT_LINES = ['camping', 'kitchen', 'ski']

# Ensure schema exists
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA_NAME}")
print(f"✅ Schema '{SCHEMA_NAME}' ready!")
print(f"📁 Loading finance data from {len(PRODUCT_LINES)} product lines: {', '.join(PRODUCT_LINES)}")

In [ ]:
# 1. Load Invoice Table from All Product Lines
print("🧾 Loading Invoice table from all product lines...")

invoice_dfs = []

for product_line in PRODUCT_LINES:
    print(f"  📦 Loading {product_line} invoices...")
    
    # Read CSV file
    invoice_df = spark.read.csv(
        f"{BASE_PATH}/{product_line}/finance/Invoice_Samples_{product_line.title()}.csv",
        header=True,
        inferSchema=True
    )
    
    print(f"     Records loaded: {invoice_df.count()}")
    invoice_dfs.append(invoice_df)

# Union all invoice dataframes
combined_invoice_df = invoice_dfs[0]
for df in invoice_dfs[1:]:
    combined_invoice_df = combined_invoice_df.union(df)

total_invoices = combined_invoice_df.count()
print(f"\n📊 Total invoices combined: {total_invoices}")

# Write to Delta table
combined_invoice_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_NAME}.invoice")

print(f"✅ Invoice table loaded successfully!")

In [ ]:
# 2. Load Account Table from All Product Lines
print("🏦 Loading Account table from all product lines...")

account_dfs = []

for product_line in PRODUCT_LINES:
    print(f"  📦 Loading {product_line} accounts...")
    
    # Read CSV file
    account_df = spark.read.csv(
        f"{BASE_PATH}/{product_line}/finance/Account_Samples_{product_line.title()}.csv",
        header=True,
        inferSchema=True
    )
    
    print(f"     Records loaded: {account_df.count()}")
    account_dfs.append(account_df)

# Union all account dataframes
combined_account_df = account_dfs[0]
for df in account_dfs[1:]:
    combined_account_df = combined_account_df.union(df)

total_accounts = combined_account_df.count()
print(f"\n📊 Total accounts combined: {total_accounts}")

# Write to Delta table
combined_account_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_NAME}.account")

print(f"✅ Account table loaded successfully!")

In [ ]:
# 3. Load Payment Table from All Product Lines
print("💰 Loading Payment table from all product lines...")

payment_dfs = []

for product_line in PRODUCT_LINES:
    print(f"  📦 Loading {product_line} payments...")
    
    # Read CSV file
    payment_df = spark.read.csv(
        f"{BASE_PATH}/{product_line}/finance/Payment_Samples_{product_line.title()}.csv",
        header=True,
        inferSchema=True
    )
    
    print(f"     Records loaded: {payment_df.count()}")
    payment_dfs.append(payment_df)

# Union all payment dataframes
combined_payment_df = payment_dfs[0]
for df in payment_dfs[1:]:
    combined_payment_df = combined_payment_df.union(df)

total_payments = combined_payment_df.count()
print(f"\n📊 Total payments combined: {total_payments}")

# Write to Delta table
combined_payment_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_NAME}.payment")

print(f"✅ Payment table loaded successfully!")

In [ ]:
# Summary and Verification
print("🎉 All finance tables loaded successfully!")
print("\n📅 Summary:")

# Show table counts
tables = ['invoice', 'account', 'payment']

for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as count FROM {SCHEMA_NAME}.{table}").collect()[0]['count']
    print(f"   • {SCHEMA_NAME}.{table}: {count:,} records")

# Show breakdown by product line for verification
print(f"\n📦 Data Distribution Verification:")
print(f"   • Loaded from {len(PRODUCT_LINES)} product lines: {', '.join(PRODUCT_LINES)}")
print(f"   • Each product line contributed Invoice, Account, and Payment data")

print(f"\n📁 All tables are available in the '{SCHEMA_NAME}' schema")
print("🚀 Ready for financial analytics and reporting!")